In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!sudo apt-get update -q
!sudo apt-get install bcftools -y -q

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,934 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,302 kB]
Ge

In [6]:
# INDEX NEW ANNOTATED FILE TO USE THEM
import os
import subprocess

VCF_PATH = "/content/drive/MyDrive/FYP_DATA/DATA/annotated_data_mane/sg10k/sg10k_annotated/sg10k_chr22_SAS_annotated_mane.vcf.gz"
tbi_file  = VCF_PATH + '.tbi'

result = subprocess.run(
    f"bcftools index -t {VCF_PATH}",
    shell=True, capture_output=True, text=True
)

if result.returncode == 0:
  tbi_size = os.path.getsize(tbi_file) / (1024**2)
  print(f"✅ Done ({tbi_size:.2f} MB index)")
else:
  print(f"❌ Failed: {result.stderr}")

✅ Done (0.03 MB index)


In [8]:
# ============================================================
# COMPREHENSIVE VCF COLUMN INSPECTOR
# Shows ALL columns: VCF standard, INFO fields, CSQ fields, FORMAT
# ============================================================

import subprocess

# ============================================================
# CONFIGURATION — Change this to inspect any VCF
# ============================================================
VCF_PATH = "/content/drive/MyDrive/FYP_DATA/DATA/annotated_data_mane/sg10k/sg10k_annotated/sg10k_chr22_SAS_annotated_mane.vcf.gz"

# ============================================================
# SECTION 1: STANDARD VCF COLUMNS
# ============================================================

print("="*80)
print(f"VCF: {VCF_PATH}")
print("="*80)

# Get column names from header
result = subprocess.run(
    f"bcftools view -h {VCF_PATH} | tail -1",
    shell=True, capture_output=True, text=True
)

vcf_columns = result.stdout.strip().split('\t')

print(f"\n📋 SECTION 1: STANDARD VCF COLUMNS ({len(vcf_columns)} total)")
print("-"*80)

# First 9 are standard
standard_cols = vcf_columns[:9]
sample_cols = vcf_columns[9:]

print(f"\nStandard columns (always present):")
for i, col in enumerate(standard_cols):
    print(f"  [{i}] {col}")

print(f"\nSample columns ({len(sample_cols)} samples):")
if len(sample_cols) <= 10:
    for col in sample_cols:
        print(f"  {col}")
else:
    print(f"  {sample_cols[0]}, {sample_cols[1]}, {sample_cols[2]}, ... {sample_cols[-1]}")
    print(f"  (showing first 3 and last, total {len(sample_cols)} samples)")

# ============================================================
# SECTION 2: INFO FIELDS
# ============================================================

print(f"\n\n📊 SECTION 2: INFO FIELDS")
print("-"*80)

result = subprocess.run(
    f"bcftools view -h {VCF_PATH} | grep '^##INFO'",
    shell=True, capture_output=True, text=True
)

if result.stdout:
    info_lines = result.stdout.strip().split('\n')
    print(f"\nFound {len(info_lines)} INFO fields:\n")

    print(f"{'Field':<20} {'Type':<10} {'Number':<10} {'Description':<50}")
    print("-"*95)

    for line in info_lines:
        # Parse INFO line: ##INFO=<ID=AF,Number=A,Type=Float,Description="...">
        if 'ID=' in line:
            field_id = line.split('ID=')[1].split(',')[0]

            field_type = ''
            if 'Type=' in line:
                field_type = line.split('Type=')[1].split(',')[0]

            field_number = ''
            if 'Number=' in line:
                field_number = line.split('Number=')[1].split(',')[0]

            description = ''
            if 'Description=' in line:
                desc_raw = line.split('Description=')[1].rstrip('>')
                description = desc_raw.strip('"').strip("'")[:47]
                if len(desc_raw) > 50:
                    description += "..."

            # Highlight important fields
            marker = ""
            if field_id in ['AF', 'AC', 'AN', 'SAS_AF', 'AR2', 'DR2', 'CSQ']:
                marker = "⭐"

            print(f"{field_id:<20} {field_type:<10} {field_number:<10} {description:<50} {marker}")
else:
    print("⚠️  No INFO fields found")

# ============================================================
# SECTION 3: CSQ ANNOTATION COLUMNS (VEP)
# ============================================================

print(f"\n\n🧬 SECTION 3: CSQ ANNOTATION COLUMNS (VEP)")
print("-"*80)

result = subprocess.run(
    f"bcftools view -h {VCF_PATH} | grep '##INFO=<ID=CSQ'",
    shell=True, capture_output=True, text=True
)

if result.stdout.strip():
    csq_line = result.stdout.strip()

    if 'Format:' in csq_line:
        format_str = csq_line.split('Format:')[1].strip().rstrip('">').strip()
        csq_fields = format_str.split('|')

        print(f"\nFound {len(csq_fields)} CSQ columns:\n")
        print(f"{'Index':<8} {'Column Name':<30} {'Notes'}")
        print("-"*80)

        for i, field in enumerate(csq_fields):
            notes = ""
            if field == 'CANONICAL':
                notes = "⭐ --canonical flag"
            elif field == 'MANE_SELECT':
                notes = "⭐ --mane flag (RefSeq ID)"
            elif field == 'MANE_PLUS_CLINICAL':
                notes = "⭐ --mane flag (additional)"
            elif field == 'Consequence':
                notes = "⭐ missense_variant, etc."
            elif field == 'IMPACT':
                notes = "⭐ MODERATE, HIGH, etc."
            elif field == 'SYMBOL':
                notes = "⭐ Gene name"
            elif field == 'Protein_position':
                notes = "⭐ Required for model"
            elif field == 'Amino_acids':
                notes = "⭐ Required for model"
            elif field == 'BIOTYPE':
                notes = "⭐ protein_coding, etc."

            print(f"[{i:>2}]      {field:<30} {notes}")
    else:
        print("⚠️  Could not parse CSQ Format")
else:
    print("⚠️  No CSQ field found (not VEP-annotated)")

# ============================================================
# SECTION 4: FORMAT FIELDS (Genotype data)
# ============================================================

print(f"\n\n🔬 SECTION 4: FORMAT FIELDS (Genotype Data)")
print("-"*80)

result = subprocess.run(
    f"bcftools view -h {VCF_PATH} | grep '^##FORMAT'",
    shell=True, capture_output=True, text=True
)

if result.stdout:
    format_lines = result.stdout.strip().split('\n')
    print(f"\nFound {len(format_lines)} FORMAT fields:\n")

    print(f"{'Field':<15} {'Type':<10} {'Description':<50}")
    print("-"*80)

    for line in format_lines:
        if 'ID=' in line:
            field_id = line.split('ID=')[1].split(',')[0]

            field_type = ''
            if 'Type=' in line:
                field_type = line.split('Type=')[1].split(',')[0]

            description = ''
            if 'Description=' in line:
                desc_raw = line.split('Description=')[1].rstrip('>')
                description = desc_raw.strip('"').strip("'")[:47]
                if len(desc_raw) > 50:
                    description += "..."

            marker = ""
            if field_id in ['GT', 'DP', 'GQ', 'AD']:
                marker = "⭐"

            print(f"{field_id:<15} {field_type:<10} {description:<50} {marker}")
else:
    print("⚠️  No FORMAT fields found")

# ============================================================
# SECTION 5: SUMMARY
# ============================================================

print(f"\n\n📝 SECTION 5: QUICK REFERENCE SUMMARY")
print("="*80)

print(f"\n🔑 KEY FIELDS FOR YOUR PROJECT:\n")

# Check what's available
has_af = any('AF' in line for line in info_lines) if result.stdout else False
has_ar2 = any('AR2' in line for line in info_lines) if result.stdout else False
has_sas_af = any('SAS_AF' in line for line in info_lines) if result.stdout else False

print(f"Population frequency:")
if has_af:
    print(f"  ✅ AF - Allele frequency (global)")
if has_sas_af:
    print(f"  ✅ SAS_AF - South Asian allele frequency")
if not has_af and not has_sas_af:
    print(f"  ⚠️  No AF fields found")

print(f"\nImputation quality:")
if has_ar2:
    print(f"  ✅ AR2 - Allelic R-squared (SG10K)")
else:
    print(f"  ⚠️  AR2 not found (expected for 1000G)")

print(f"\nVEP annotation:")
if 'csq_fields' in locals():
    if 'MANE_SELECT' in csq_fields:
        print(f"  ✅ MANE_SELECT - Annotated with --mane")
    elif 'CANONICAL' in csq_fields:
        print(f"  ⚠️  CANONICAL only - Annotated with --canonical")

    if 'Protein_position' in csq_fields:
        print(f"  ✅ Protein_position - Available")
    if 'Amino_acids' in csq_fields:
        print(f"  ✅ Amino_acids - Available")

print(f"\nSamples:")
print(f"  {len(sample_cols):,} samples in VCF")

print("\n" + "="*80)

VCF: /content/drive/MyDrive/FYP_DATA/DATA/annotated_data_mane/sg10k/sg10k_annotated/sg10k_chr22_SAS_annotated_mane.vcf.gz

📋 SECTION 1: STANDARD VCF COLUMNS (1134 total)
--------------------------------------------------------------------------------

Standard columns (always present):
  [0] #CHROM
  [1] POS
  [2] ID
  [3] REF
  [4] ALT
  [5] QUAL
  [6] FILTER
  [7] INFO
  [8] FORMAT

Sample columns (1125 samples):
  WHB1003, WHB1007, WHB1008, ... WHH999
  (showing first 3 and last, total 1125 samples)


📊 SECTION 2: INFO FIELDS
--------------------------------------------------------------------------------

Found 7 INFO fields:

Field                Type       Number     Description                                       
-----------------------------------------------------------------------------------------------
AF                   Float      A          Estimated ALT Allele Frequencies                   ⭐
AR2                  Float      1          Allelic R-Squared: estimated squ